# CryptoUPI — Fraud Detection Model

**Project:** CryptoUPI (Crypto Wallet → UPI payment bridge)
**Author:** [Aapka naam yaha likhein]
**Purpose:** Demonstrate a supervised machine learning approach to flagging
potentially fraudulent crypto-to-UPI transactions.

### Why synthetic data?
CryptoUPI is currently a student MVP, not a live production service, so no
real transaction/fraud history exists yet. A synthetic dataset is generated
instead, with realistic feature relationships based on known fraud patterns
in payments and AML (Anti-Money Laundering) literature. This is standard
academic practice when real-world labeled data is unavailable.

### Features used
| Feature | Description |
|---|---|
| `amount_inr` | Transaction size in INR |
| `hour_of_day` | Hour (0-23) the transaction was made |
| `is_new_recipient` | 1 if this UPI ID has never been paid before |
| `tx_count_last_24h` | Number of transactions by this wallet in the last 24h |
| `wallet_age_days` | Age of the sending wallet in days |
| `amount_deviation_ratio` | This amount ÷ the user's typical amount |
| `is_structured_amount` | 1 if amount is just under the ₹50,000 RBI/PMLA reporting threshold ("structuring") |

Label: `is_fraud` (0 = legitimate, 1 = fraudulent)


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score
)

np.random.seed(42)


## 2. Generate the synthetic dataset

We build a hidden "risk score" from known fraud indicators, add some
randomness, and then sample fraud labels from it. This keeps the patterns
learnable but not perfectly separable — just like real fraud data.

In [ ]:
N = 6000

amount_inr = np.random.lognormal(mean=7.5, sigma=0.9, size=N).round(2)
amount_inr = np.clip(amount_inr, 50, 200000)

hour_of_day = np.random.randint(0, 24, size=N)
is_new_recipient = np.random.binomial(1, 0.25, size=N)
tx_count_last_24h = np.random.poisson(1.2, size=N)

wallet_age_days = np.random.exponential(scale=120, size=N).round(0)
wallet_age_days = np.clip(wallet_age_days, 0, 1500)

user_avg_amount = np.random.lognormal(mean=7.3, sigma=0.6, size=N)
amount_deviation_ratio = (amount_inr / user_avg_amount).round(2)

is_structured_amount = ((amount_inr > 47000) & (amount_inr < 50000)).astype(int)

risk_score = (
    2.0 * is_new_recipient
    + 1.8 * (tx_count_last_24h >= 3).astype(int)
    + 2.2 * (wallet_age_days < 3).astype(int)
    + 2.0 * (amount_deviation_ratio > 4).astype(int)
    + 2.5 * is_structured_amount
    + 0.8 * ((hour_of_day >= 1) & (hour_of_day <= 4)).astype(int)
    + 0.6 * (amount_inr > 80000).astype(int)
)

noise = np.random.normal(0, 0.3, size=N)
fraud_prob = 1 / (1 + np.exp(-(risk_score - 3.2 + noise)))
is_fraud = np.random.binomial(1, fraud_prob)

df = pd.DataFrame({
    'amount_inr': amount_inr,
    'hour_of_day': hour_of_day,
    'is_new_recipient': is_new_recipient,
    'tx_count_last_24h': tx_count_last_24h,
    'wallet_age_days': wallet_age_days,
    'amount_deviation_ratio': amount_deviation_ratio,
    'is_structured_amount': is_structured_amount,
    'is_fraud': is_fraud,
})

print("Dataset shape:", df.shape)
print("\nFraud class balance:")
print(df['is_fraud'].value_counts(normalize=True).round(4))
df.head()


## 3. Train / test split

We hold out 20% of the data to evaluate the model on transactions it has never seen. `stratify=y` keeps the same fraud/legit ratio in both splits, which matters because fraud is a minority class.

In [ ]:
FEATURES = [
    'amount_inr', 'hour_of_day', 'is_new_recipient', 'tx_count_last_24h',
    'wallet_age_days', 'amount_deviation_ratio', 'is_structured_amount'
]
X = df[FEATURES]
y = df['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training on {len(X_train)} transactions, testing on {len(X_test)}")


## 4. Train a Random Forest classifier

A Random Forest is an ensemble of many decision trees, each trained on a random subset of the data and features, whose votes are combined. It handles non-linear patterns well and gives interpretable feature importances — useful for explaining *why* a transaction was flagged.

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=10,
    class_weight='balanced',   # compensates for fraud being a minority class
    random_state=42,
)
model.fit(X_train, y_train)


## 5. Evaluate the model

Key metrics for a fraud detector:
- **Precision** — of the transactions we flagged, how many were truly fraud (false alarms cost user trust/support effort)
- **Recall** — of all real frauds, how many did we catch (missed fraud costs real money)
- **ROC AUC** — overall ability to rank fraud higher than legitimate transactions, regardless of threshold

In fraud detection it's common to accept lower precision in exchange for higher recall — missing real fraud is usually costlier than a false alarm that a human quickly reviews.

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print(f"Accuracy : {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall   : {recall:.3f}")
print(f"F1 score : {f1:.3f}")
print(f"ROC AUC  : {auc:.3f}\n")
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))


### Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(cm, cmap='Purples')
ax.set_xticks([0, 1]); ax.set_xticklabels(['Predicted Legit', 'Predicted Fraud'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['Actual Legit', 'Actual Fraud'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                 color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=14, fontweight='bold')
ax.set_title('Confusion Matrix')
plt.colorbar(im)
plt.tight_layout()
plt.show()


### Which features matter most?

In [ ]:
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(6, 4.5))
importances.plot(kind='barh', ax=ax, color='#5B3DF6')
ax.set_title('Feature importance')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()


### ROC curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
fig, ax = plt.subplots(figsize=(5, 4.5))
ax.plot(fpr, tpr, color='#0E9F6E', linewidth=2, label=f'ROC curve (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random guess')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve')
ax.legend()
plt.tight_layout()
plt.show()


## 6. Try it on example transactions

A small helper function that scores a single transaction, the way the
CryptoUPI app would call it before confirming a payment.

In [ ]:
def predict_risk(amount_inr, hour_of_day, is_new_recipient, tx_count_last_24h,
                  wallet_age_days, amount_deviation_ratio, is_structured_amount):
    row = pd.DataFrame([{
        'amount_inr': amount_inr, 'hour_of_day': hour_of_day,
        'is_new_recipient': is_new_recipient, 'tx_count_last_24h': tx_count_last_24h,
        'wallet_age_days': wallet_age_days, 'amount_deviation_ratio': amount_deviation_ratio,
        'is_structured_amount': is_structured_amount,
    }])
    proba = model.predict_proba(row)[0, 1]
    return round(proba * 100, 1)

print("Normal small payment to a saved contact, daytime:")
print(f"  Fraud risk: {predict_risk(1500, 14, 0, 1, 400, 1.1, 0)}%")

print("\nLarge payment, new recipient, brand-new wallet, 2 AM, near Rs 50k threshold:")
print(f"  Fraud risk: {predict_risk(48500, 2, 1, 4, 1, 6.0, 1)}%")


## 7. Save the trained model

So it can be reused later without retraining.

In [ ]:
import joblib
joblib.dump(model, 'fraud_model.joblib')
print("Saved to fraud_model.joblib")


## 8. Limitations & honest disclosures

- **Synthetic data**: labels come from a hand-designed formula, not real
  observed fraud. Real fraud patterns may differ and would need to be
  learned from actual transaction history once the product has real users.
- **Simplified fraud rate**: real-world fraud is usually much rarer
  (often <1-2% of transactions) than the ~18% used here. A higher rate was
  chosen so the model has enough positive examples to learn from during
  a course project; production systems would need techniques for extreme
  class imbalance (e.g. SMOTE, anomaly detection instead of classification).
- **Not connected to the live app**: this model runs separately in Python.
  Wiring it into the actual CryptoUPI web app would require a backend
  service (e.g. a small Flask/FastAPI API) that the Next.js frontend calls
  before confirming a payment — a natural "V2" extension of this project.
